# Data check

Load a configured dataset and verify the series-to-event conversion semantics.

In [1]:
from pathlib import Path
import sys

# Use local package when running from a fresh checkout.
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))

In [2]:
from afc_robustness.experiment import BenchmarkConfig, load_dataset_from_config
from afc_robustness.data import describe_dataset
from afc_robustness.representations import series_to_episode

config_path = repo_root / "configs" / "tep.yaml"  # smoke, fcc, tep
cfg = BenchmarkConfig.from_yaml(config_path)

Generate or place data before running this cell. For a smoke dataset, run `python scripts/create_synthetic_dataset.py --output data/smoke` from the repository root.

In [3]:
dataset = load_dataset_from_config(cfg)
print(dataset)
describe_dataset(dataset)

AlarmSeriesDataset(X=array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 1],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       ...,

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 1, ..., 1, 1, 1],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 1, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 1, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
  

,class_label,class_name,n_episodes,min_length,max_length,mean_length
0,0,idv1,200,60,60,60.0
1,1,idv14,200,60,60,60.0
2,2,idv1_and_idv5,200,60,60,60.0
3,3,idv2,200,60,60,60.0
4,4,idv6,200,60,60,60.0


In [4]:
idx = 0
episode = dataset.episode(idx)
print("sample_id:", episode.sample_id)
print("n_events:", episode.n_events)
print("horizon:", episode.horizon)
print("first events:", episode.events[:10])

sample_id: idv1/alarm_timeseries_1
n_events: 33
horizon: 59.0
first events: (AlarmEvent(tag=17, event_type=<EventType.ACT: 'ACT'>, timestamp=3.0, order=11), AlarmEvent(tag=17, event_type=<EventType.RTN: 'RTN'>, timestamp=4.0, order=12), AlarmEvent(tag=47, event_type=<EventType.ACT: 'ACT'>, timestamp=11.0, order=29), AlarmEvent(tag=47, event_type=<EventType.RTN: 'RTN'>, timestamp=12.0, order=30), AlarmEvent(tag=17, event_type=<EventType.ACT: 'ACT'>, timestamp=21.0, order=13), AlarmEvent(tag=17, event_type=<EventType.RTN: 'RTN'>, timestamp=23.0, order=14), AlarmEvent(tag=32, event_type=<EventType.ACT: 'ACT'>, timestamp=29.0, order=20), AlarmEvent(tag=32, event_type=<EventType.RTN: 'RTN'>, timestamp=30.0, order=21), AlarmEvent(tag=17, event_type=<EventType.ACT: 'ACT'>, timestamp=31.0, order=15), AlarmEvent(tag=17, event_type=<EventType.RTN: 'RTN'>, timestamp=32.0, order=16))


In [5]:
n_episodes = dataset.n_episodes
act_counts = []

for i in range(n_episodes):
    ep = dataset.episode(i)
    n_act = sum(getattr(event.event_type, "value", event.event_type) == "ACT" for event in ep.events)
    act_counts.append(n_act)

total_act_events = sum(act_counts)
min_act = min(act_counts)
max_act = max(act_counts)
mean_act = total_act_events / n_episodes

print(f"Number of episodes: {n_episodes}")
print(f"Total alarm activation events: {total_act_events}")
print(f"Activation events per episode range: {min_act} to {max_act}")
print(f"Mean activation events per episode: {mean_act:.2f}")

Number of episodes: 1000
Total alarm activation events: 30019
Activation events per episode range: 9 to 72
Mean activation events per episode: 30.02
